In [20]:
import pandas as pd
import numpy as np
from IPython.display import display
df = pd.read_csv(
    "/content/store_data.csv",
    sep=",",
    encoding="latin-1"
)

In [21]:
df.drop_duplicates(subset=["Row ID"], inplace=True)

In [22]:
duplicate_mask = df["Order ID"].duplicated(keep=False)

df["Panier"] = (
    df.groupby("Order ID")["Product Name"]
      .transform(lambda products: list(products))
)

In [23]:
df['Order Date'] = pd.to_datetime(df["Order Date"],format="mixed")
df['Ship Date'] = pd.to_datetime(df["Ship Date"],format="mixed")
display(df[['Order Date', 'Ship Date']].dtypes)


,0
Order Date,datetime64[ns]
Ship Date,datetime64[ns]


In [24]:
invalid_date = df["Order Date"] > df["Ship Date"]
df["shipping_delay"] = df["Ship Date"] - df["Order Date"]
valid_delay = df.loc[~invalid_date, "shipping_delay"]
median_delay = valid_delay.median()
display("median: ",median_delay)
df.loc[invalid_date, "Ship Date"] = (
    df.loc[invalid_date, "Order Date"] + median_delay
)

'median: '

Timedelta('4 days 00:00:00')

In [25]:
mode_ship_mode = df["Ship Mode"].mode()[0]
df["Ship Mode"] = df["Ship Mode"].fillna(mode_ship_mode)

In [26]:
df["Customer ID"] = df["Customer ID"].str.strip().str.upper()

In [27]:
freconcy_name = df["Customer Name"].mode()[0]
df["Customer Name"] = df["Customer Name"].fillna(freconcy_name)

In [28]:
category_corrections = {
    "Home Ofice": "Home Office",
    "Consumerr": "Consumer",
    "Corporrate": "Corporate"
}

df["Segment"] = df["Segment"].replace(category_corrections)

In [29]:
df["City"] = (df["City"].astype("string").str.strip().str.lower().str.title())
df["State"] = (df["City"].astype("string").str.strip().str.lower().str.title())

In [30]:
missing_postal = df["Postal Code"].isna()

postal_map_state = (
    df.dropna(subset=["Postal Code"])
      .groupby("State")["Postal Code"]
      .agg(lambda x: x.mode()[0])
)

df.loc[missing_postal, "Postal Code"] = (
    df.loc[missing_postal, "State"].map(postal_map_state)
)

In [31]:
# df["Product ID"].astype("string").str.match(r"^[A-Z]{3}-[A-Z]{2}-\d{8}$", na=False).sum()

In [32]:
categorys = {
    "furniture":"Furniture",
    "office supplies":"Office Supplies",
    "technology":"Technology"
}
df["Category"] = df["Category"].replace(categorys)

In [33]:
q1 = df["Sales"].quantile(0.25)
q3 = df["Sales"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers_mask = (df["Sales"] < lower) | (df["Sales"] > upper)

inliers = df.loc[~outliers_mask, "Sales"]

inliers_median = inliers.median()

df.loc[outliers_mask, "Sales"] = inliers_median

In [34]:
outlier_mask = df["Quantity"] < 0
inlier_quantity = df.loc[~outlier_mask, "Quantity"]
inlier_median = inlier_quantity.median()
df.loc[outlier_mask, "Quantity"] = inlier_median

In [35]:
q1 = df["Discount"].quantile(0.25)
q3 = df["Discount"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers_mask = (df["Discount"] < lower) | (df["Discount"] > upper)

inliers = df.loc[~outliers_mask, "Discount"]

inliers_median = inliers.median()

df.loc[outliers_mask, "Discount"] = inliers_median

In [36]:
q1 = df["Profit"].quantile(0.25)
q3 = df["Profit"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers_mask = (df["Profit"] < lower) | (df["Profit"] > upper)

inliers = df.loc[~outliers_mask, "Profit"]

inliers_median = inliers.median()

df.loc[outliers_mask, "Profit"] = inliers_median